# 🤖 tina4-harness — Fact Sheet

**A teaching tool that shows how an agent harness actually works** — a JupyterLab extension + a
kernel-side Python "brain" driving a **small local model, fully offline**. No API keys, no cloud.

> The point isn't the model — it's the *harness*: how it routes intent, grounds answers in reality,
> checks its own code by running it, and defers honestly when it's stuck.

## What it is

- **Frontend:** a JupyterLab prebuilt extension (`labext/`, TypeScript) — the chat panel + cell actions.
- **Brain:** a kernel-side Python harness (`src/harness/`) — routing, grounding, validation.
- **Model:** one **shared** local model server (`:7777`); every notebook kernel is a thin proxy over HTTP,
  so the model lives in RAM **once** no matter how many notebooks are open.
- **Offline:** model weights + docs are bundled; nothing leaves the machine.

## The model

| | Default (2B) | Swap-in (Gemma) |
|---|---|---|
| Model | Qwen3.8-2B-Distill, 4-bit MLX | Gemma-4-E4B, 4-bit MLX |
| Weights in RAM | ~1.6 GB | ~4.8 GB |
| Peak (generation) | ~3.0 GB | ~5.7 GB |
| Speed | ~1.7 s/task | ~2.7 s/task |
| Vision + text | same weights | same weights |

Both pass the same quality suite. The **2B is the laptop default**; swap to Gemma with one env var:

```bash
HARNESS_MLX_REPO=mlx-community/gemma-4-e4b-it-4bit ./scripts/start.sh
```

## 💾 Memory use (measured, live on this machine)

Default **2B** model, on Apple Silicon (MLX / Metal unified memory):

| Component | RAM |
|---|---|
| Model weights (resident) | **1.6 GB** |
| Peak during a generation | **~3.0 GB** |
| KV / compute cache | ~0.09 GB (capped at 1.0) |
| JupyterLab server | ~145 MB |
| Each open notebook kernel | **~95 MB** (a thin proxy — no model copy) |
| On disk | 1.6 GB |

**Why the shared server matters:** open 5 notebooks and you still pay the **1.6 GB once** — each
kernel is a `RemoteModel` proxy over HTTP, not its own copy of the weights. Swap to Gemma and the
same design holds: ~4.8 GB resident, ~5.7 GB peak — ~3× the weights for comparable quality, which
is exactly the trade-off the fact sheet lets you show.

## What the chat can do

- **Ask / explain** — short by default; say **"teach me X"** for a build-up explanation (no code dump).
- **Write code** — generated into a cell and run. Code it *can't safely check* (network/file) is
  inserted but **not auto-run** — you run it when ready.
- **Fix / edit in place** — edits the selected cell (with Undo), never a stray new cell.
- **Debug** — "what's wrong?" **runs the cell with sample inputs** and shows the real behaviour,
  then offers a fix.
- **Test** — "add a function and a test" → a **pytest-style test that asserts the *expected* value**
  (a wrong implementation fails the test, instead of hiding the bug).
- **Vision** — ask about an open image, or "show me the code in this screenshot".
- **Cells** — target "cell 3", delete a cell, clear all / run all, multi-cell.

## 🔧 Methods — how it works under the hood

The techniques that make a *2B* model dependable inside the harness:

- **Output-form routing.** The model reliably writes a *code block* for a code request and *prose*
  for a question, so the harness reads the **form of the answer** to decide ask-vs-code — a UI
  misroute self-corrects. (A model *classifier* was tried and abandoned — a 2B answers instead of labelling.)
- **Render sentinels.** Tiny `@@HARNESS_*@@` markers the kernel prepends tell the extension how to
  render: insert a cell, offer a fix, show choice pills, or *insert-but-don't-run*.
- **Self-check by running.** Generated code is executed in an isolated copy of the kernel namespace
  *before* it's pasted; on a crash it grounds the error and retries, then defers honestly if stuck.
- **Fix validation.** It runs the original vs the proposed code and compares **behaviour** (AST-based,
  docstrings stripped) — only a change that actually fixes/alters behaviour is offered.
- **Grounding (offline BM25 / Groundwire).** Answers are grounded in bundled docs (Python stdlib,
  tina4 python+js), your working-folder code, and **live module introspection** on an import error
  (so it won't invent `antigravity.Client` or suggest installing a module that's already present).
- **Conversation memory.** A rolling, model-free **digest** of earlier turns + **BM25-retrieved**
  relevant older turns keeps a long chat coherent within a bounded prompt.
- **Repetition penalty (1.18).** Stops the small model degenerating into a loop.
- **Vision = resolution, not format.** Images are LANCZOS-resized to the model's pixel sweet spot
  (~1.2M px for the 2B; Gemma reads fine text at far lower res) — the real quality lever.
- **Model-agnostic reasoning strip.** Handles Qwen's `<think>…</think>` AND Gemma's
  `<|channel>thought…<channel|>`, so reasoning never leaks into the answer.
- **Chat-safe markdown.** Headings → bold, tables → bullet lists, no ASCII-art — so replies read
  cleanly in a narrow chat bubble.

## How it stays honest (the harness thesis)

- **Routes by understanding**, not keyword whack-a-mole.
- **Grounds before it guesses** — docs, your code, real modules.
- **Checks itself** — runs generated code before pasting; retries with the real error.
- **Defers honestly** — hands back its best attempt + the real error and asks, rather than pasting junk.
- **Asks when unsure** — an ambiguous request gets one short question, not a wrong guess.

## Run it

```bash
./scripts/start.sh      # shared model server (:7777) + JupyterLab (:8890)
```

Tests (real — no mocks): `PYTHONPATH=src .venv-notebook/bin/python -m pytest tests/ -q`

---
## 🎬 Demo script — try these in the chat panel →

**1. Teaching**  →  `teach me python list comprehensions`  *(explains, no code dump)*

**2. Write code**  →  `write a function to reverse a string`  *(self-checked, then run)*

**3. Test generation**  →  `add 2 cells - a function to add two numbers and a pytest test for it`

**4. Debug** — click the buggy cell below, then  →  `what's wrong here?`

**5. Safe-by-default**  →  `write code to fetch https://example.com with requests`  *(inserted, NOT auto-run)*

**6. Vision** — open an image tab, then  →  `what's happening in this image?`

### Demo cell A — a correct function (try: *"write a test for this function"*)

In [ ]:
def word_count(sentence: str) -> int:
    """Count the words in a sentence."""
    return len(sentence.split())

word_count("the quick brown fox")

### Demo cell B — a *buggy* function (try: *"what's wrong here?"*)

_It's named `add` but returns `a + 1` — the harness will run it, show `add(2,3) -> 3`, and flag it._

In [ ]:
def add(a, b):
    return a + 1

add(2, 3)

---
*Generated as a demo reference. The harness is a teaching project — everything here runs on a local
model, offline. 🤖*